# System performance, small check on a T4: Qwen3-VL-4B, one video, all its questions (Colab T4)

A quick run of `system_perf/` on a **T4** (16 GB, no bf16, so the model runs in **fp16**), **Qwen3-VL-4B only**: the other models are commented out in the settings cell.
The test video `loc2_script1_seq2_rec1` with **all its 23 questions**, every method and condition of Qwen3's budget-matched
config (made by step 0). Per method and condition: **TTFT** (vision side + prefill), **TPS** (64 greedy tokens), **VE / LD / overall peak dynamic
memory**, **frames encoded**, **vision-side GFLOPs**, **method overhead**, **energy and power** (NVML), decoder **memory vs batch
size** (1-8). The language decoder's GFLOPs are its prefill; combined = vision side + prefill. One warm-up per condition (the A100 notebook uses 3). Details: `system_perf/README.md`.

| `MODEL` | conditions from | fp16 on the T4 |
|---|---|---|
| `qwen3` | `accuracy_tables/qwen3_matched_t4/config_matched.yaml`, **made by step 0 here**: it has no budget-matched run | `system_perf/t4.py` converts it, in this notebook's processes only |

Every step runs through `python -m system_perf.t4`. On a T4 it converts Qwen3-VL to fp16 in that process only; nothing in
`vlmeval/` or `accuracy_tables/` changes, so the cluster's jobs don't either. On a GPU with bf16 it changes nothing.

**Expect:**
- **fp16 can overflow** where bf16 doesn't. Parity at the start of every step, and `nan` records, show it. Never use `--no-parity`.
- **Uniform K's 96 frames may run out of memory** on 16 GB, its prompt being about 25,000 tokens: a question's `oom` error (on the first question, its
  FLOPs `NOT COUNTED, oom` too), or `decode_error: oom` with no TPS. **Batch sizes above 1 will mostly be `oom`.** These are
  recorded and the run goes on; the report leaves them out. Budgets and speed-ups are vs Uniform K, so if Uniform K runs out of
  memory on every question they show `--`, and step 0's calibrate stops (it needs Uniform K's token counts).
- **Qwen3's conditions come from this T4,** in fp16, so FastVID's and PruneVid's knobs may differ slightly from an A100 calibration.
  There is no Qwen3 accuracy in the report: its baselines aren't scored for accuracy here.

Results go to `MyDrive/GazeVLM/system_perf_t4/qwen3/` (separate from the A100 results). Rough time on a T4, an estimate:
about 2 hours for step 0, then about 3 hours.

In [ ]:
import os, sys, subprocess
import torch

# A small check on a T4 (fp16); any CUDA GPU works, the A100 notebook is the full protocol.
gpu = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
print('GPU:', f'{gpu.name}, {gpu.total_memory / 1e9:.0f} GB' if gpu else 'none')
assert gpu, 'needs a GPU runtime: Runtime > Change runtime type > T4 GPU'

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
# num2words: needed by SmolVLM2's processor (install before transformers is first imported)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pytest', 'num2words', 'pyyaml', 'nvidia-ml-py'],
               check=True)

In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')
try:                                         # Hugging Face token from the Colab secret HF_TOKEN
    from google.colab import userdata
    from huggingface_hub import login
    if userdata.get('HF_TOKEN'):
        login(token=userdata.get('HF_TOKEN'))
except Exception as ex:
    print('no HF_TOKEN secret:', type(ex).__name__)

MODEL, LABEL = 'qwen3', 'Qwen3-VL-4B'             # this notebook: Qwen3-VL only
# MODEL, LABEL = 'smolvlm', 'SmolVLM2-2.2B'       # other models, commented out
# MODEL, LABEL = 'qwen2.5', 'Qwen2.5-VL-3B'
VIDEO = 'loc2_script1_seq2_rec1'     # a test video: 23 questions
DRIVE = '/content/drive/MyDrive/GazeVLM'
ACC = f'{DRIVE}/accuracy_tables/qwen3_matched_t4'                        # made by step 0 below, on this T4
# ACC = f'{DRIVE}/accuracy_tables/smolvlm_matched'                       # SmolVLM2: its A100 budget-matched accuracy run
# ACC = f'{DRIVE}/accuracy_tables/qwen25_matched'                        # Qwen2.5: its A100 budget-matched accuracy run
MATCHED = f'{ACC}/config_matched.yaml'
SYS = f'{DRIVE}/system_perf_t4'                                           # separate from the A100 results
QUESTIONS, WARMUP, BATCH = 1000, 1, '1,2,4,8'                            # 1000 = all of the video's questions
RUN = 'python -m system_perf.t4'                                          # Qwen3-VL in fp16 on a T4 (system_perf/t4.py)
CONFIG = 'accuracy_tables/config.yaml'                                    # step 0's config, as the matched notebooks use it
if not os.path.exists(CONFIG):
    shutil.copy('accuracy_tables/config.example.yaml', CONFIG)
# if MODEL != 'qwen3':                                                   # other models only
#     assert os.path.exists(MATCHED), f'run the budget-matched {LABEL} accuracy notebook first ({MATCHED} missing)'
import transformers                                                      # Qwen3-VL needs transformers >= 4.57
print('transformers', transformers.__version__)
assert hasattr(transformers, 'Qwen3VLForConditionalGeneration'), \
    'this transformers has no Qwen3-VL (needs >= 4.57): pip install -U transformers, then restart the runtime'
print(LABEL, '| conditions from', MATCHED, '-> results in', f'{SYS}/{MODEL}')

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_system_perf.py', 'tests/test_system_perf_t4.py',
                    'tests/test_accuracy_tables.py', '-q'], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, 'unit tests failed'

## Step 0: Qwen3's conditions

Qwen3-VL has no budget-matched accuracy run, so this makes its `config_matched.yaml` the way the A100 matched notebooks do:
Uniform K and ours on the 4 test videos, then `calibrate.py` sets every baseline's knob to ours' 2 smallest measured budgets, from
token counts only. Skipped if the file already exists. Parity runs first.

In [ ]:
if MODEL == 'qwen3' and not os.path.exists(MATCHED):
    os.makedirs(ACC, exist_ok=True)
    get_ipython().system(f'{RUN} accuracy_tables.run --config {CONFIG} --model {MODEL} --rows uniform,ours '
                         f'--videos test --out-dir {ACC}')
    get_ipython().system(f'{RUN} accuracy_tables.calibrate --config {CONFIG} --model {MODEL} --points 2 '
                         f'--videos test --out-dir {ACC} --write {MATCHED}')
assert os.path.exists(MATCHED), f'no {MATCHED}: see the output above'
print(open(MATCHED).read()[:3000])

## Measure

Parity first, then every row and condition of the matched config on the sampled questions. Each line printed is one
question: TTFT (vision side), TPS, VE / LD peak memory, frames encoded, visual tokens. Rerunning appends; the report
keeps the last record per question.

In [ ]:
get_ipython().system(f'{RUN} system_perf.measure --config {MATCHED} --model {MODEL} --videos {VIDEO} '
                     f'--questions {QUESTIONS} --warmup {WARMUP} --dtype auto --out-dir {SYS} --batch-sizes {BATCH}')

## Report

Mean +- s.d. per condition, speed-ups and memory savings vs Uniform K, method overheads, no accuracy (Qwen3's baselines aren't
scored here); `system_table.csv` and `system_table.tex`.

In [ ]:
ACCURACY = f'{ACC}/summary.csv'
extra = f'--accuracy {ACCURACY} ' if os.path.exists(ACCURACY) else ''
get_ipython().system(f'{RUN} system_perf.report --results {SYS} --model {MODEL} {extra}--label "{LABEL}"')
print(open(f'{SYS}/{MODEL}/system_table.tex').read())